In [18]:
!git clone https://github.com/guanjq/targetdiff.git

Cloning into 'targetdiff'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 163 (delta 35), reused 30 (delta 30), pack-reused 109 (from 1)
Receiving objects: 100% (163/163), 10.44 MiB | 17.41 MiB/s, done.
Resolving deltas: 100% (59/59), done.


In [22]:
import os
# The correct folder name is lowercase 'targetdiff'
os.chdir('/content/targetdiff')
!pwd

/content/targetdiff


In [23]:
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
!pip install rdkit openbabel pyyaml easydict meeko

Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu121.html
  Using cached rdkit-2026.3.3-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.8 kB)
  Using cached openbabel-3.2.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 30.1 MB/s eta 0:00:00


In [28]:
!find / -name "3M4I_perfect_clean.pdb" 2>/dev/null

/content/targetdiff/3M4I_perfect_clean.pdb


In [29]:
!mv /content/targetdiff/3M4I_perfect_clean.pdb

mv: missing destination file operand after '/content/targetdiff/3M4I_perfect_clean.pdb'
Try 'mv --help' for more information.


In [ ]:
import pickle
import torch
from datasets import get_dataset
import utils.misc as misc
import utils.transforms as trans
from torch_geometric.transforms import Compose
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
from models.molopt_score_model import ScorePosNet3D
from scripts.likelihood_est_diffusion import data_likelihood_estimation

In [ ]:
sampling_config = 'configs/sampling.yml'
sampling_config = misc.load_config(sampling_config)

In [ ]:
# Load checkpoint
device = 'cuda:0'
ckpt = torch.load(sampling_config.model.checkpoint, map_location=device)

# Transforms
protein_featurizer = trans.FeaturizeProteinAtom()
ligand_atom_mode = ckpt['config'].data.transform.ligand_atom_mode
ligand_featurizer = trans.FeaturizeLigandAtom(ligand_atom_mode)
transform = Compose([
    protein_featurizer,
    ligand_featurizer,
    trans.FeaturizeLigandBond(),
])

In [ ]:
# Load model
model = ScorePosNet3D(
    ckpt['config'].model,
    protein_atom_feature_dim=protein_featurizer.feature_dim,
    ligand_atom_feature_dim=ligand_featurizer.feature_dim
).to(device)
model.load_state_dict(ckpt['model'], strict=True)
print(f'Successfully load the model! {sampling_config.model.checkpoint}')

Successfully load the model! ./pretrained_models/pretrained_diffusion.pt


In [ ]:
from utils.data import PDBProtein, parse_sdf_file
from datasets.pl_data import ProteinLigandData, torchify_dict
from torch.utils.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch

In [ ]:
def convert_data(pdb_path, ligand_path, transform, radius=10, pocket=False):
    # ligand_dict = parse_sdf_file_mol(ligand_path, heavy_only=False)
    ligand_dict = parse_sdf_file(ligand_path)
    if not pocket:
        protein = PDBProtein(pdb_path)
        pdb_block_pocket = protein.residues_to_pdb_block(
            protein.query_residues_ligand(ligand_dict, radius)
        )
        pocket_dict = PDBProtein(pdb_block_pocket).to_dict_atom()
    else:
        pocket_dict = PDBProtein(pdb_path).to_dict_atom()

    data = ProteinLigandData.from_protein_ligand_dicts(
        protein_dict=torchify_dict(pocket_dict),
        ligand_dict=torchify_dict(ligand_dict),
    )
    data.protein_filename = pdb_path
    data.ligand_filename = ligand_path
    # data.y = torch.tensor(float(pka))
    # data.kind = torch.tensor(2)  # kd
    # data.id = idx
    assert data.protein_pos.size(0) > 0
    if transform is not None:
        data = transform(data)
    return data

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, data_list):
        super().__init__()
        self.data_list = data_list

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        data = self.data_list[idx]
        return data

In [ ]:
test_data = convert_data('examples/3ug2_protein.pdb',
                         'examples/3ug2_ligand.sdf', transform)
# test_data.kind = KMAP[args.kind]
test_set = InferenceDataset([test_data])
test_loader = DataLoader(test_set, batch_size=1, shuffle=False, follow_batch=['protein_element', 'ligand_element'])

In [ ]:
batch = next(iter(test_loader)).to(device)

In [ ]:
preds = model.fetch_embedding(
            protein_pos=batch.protein_pos,
            protein_v=batch.protein_atom_feature.float(),
            batch_protein=batch.protein_element_batch,

            ligand_pos=batch.ligand_pos,
            ligand_v=batch.ligand_atom_feature_full,
            batch_ligand=batch.ligand_element_batch,
        )

In [ ]:
# load linear model
with open('pretrained_models/pk_reg_para.pkl', 'rb') as f:
    lmodel = pickle.load(f)

In [ ]:
final_ligand_h = np.array([preds['final_ligand_h'].cpu().numpy().mean(0)])

In [ ]:
pka = lmodel.predict(final_ligand_h)

In [ ]:
affinity = np.power(10, -pka)

In [ ]:
affinity

array([1.0119374e-09], dtype=float32)